[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C06_Interpretability_Course/04_activation_patching/04_activation_patching.ipynb)

# 04 · Activation Patching：因果干预定位行为来源

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy + matplotlib，无 torch、无下载、秒级运行。

**本 notebook 你将完成：**

1. **手工构造**一个 2 层 × 2 头的 numpy transformer，零训练解出**键值召回**任务（`[BOS, k1, v1, k2, v2, k3, v3, kq]` → 在末位置输出 `kq` 配对的 value），电路完全透明；
2. 定义 **clean / corrupted 最小对**（换 value、换 query、换 key 三种 corruption）与 **logit diff** 度量，跑通 clean / corrupted / patched 三次前向协议；
3. 实现**逐层逐位置 residual stream patching**，画 (layer × position) 因果热图——亲眼看见差异信息"先驻留在 value 位置、再被搬到末位置"的轨迹（causal tracing 微缩版）；
4. 实现 **head-level patching**，从 4 个 head 中定位出串成一条因果路径的 2 个关键 head；
5. 在手工**冗余电路**上精确复现 noising 与 denoising 的不对称（同一位置 denoising recovery = 1.0、noising effect = 0.0），并看到 softmax 重归一化导致的 **self-repair**；
6. 4 道 ✏️ 练习巩固以上全部核心函数。

参考：[Meng 2022] *Locating and Editing Factual Associations in GPT*（ROME, arXiv:2202.05262）、[Wang 2022] *Interpretability in the Wild*（IOI, arXiv:2211.00593）、[Zhang & Nanda 2023] *Towards Best Practices of Activation Patching*（arXiv:2309.16042）、[Heimersheim & Nanda 2024] *How to Use and Interpret Activation Patching*（arXiv:2404.15255）。


## 1 · 手工构造：会做键值召回的 2 层 transformer

任务（**键值召回 key-value recall**）：序列 `[BOS, k1, v1, k2, v2, k3, v3, kq]`——context 里写三对 key→value，末位置再给一个 query key，模型要在末位置输出它配对的 value。key 取 token 1–7、value 取 8–15，**取值区间不相交**，匹配无歧义。

residual stream 沿用模块 02/03 的分块布局，电路肉眼可读：

```
[ TOK | POS | KEY | OUT ]
  TOK: 当前 token one-hot（embedding 写入，之后无人覆盖）
  POS: 位置 one-hot
  KEY: Layer 1 prev-token head 写入——"本位置的前一个 token"
  OUT: Layer 2 写入——unembedding W_U 只读这一块
```

两层各 2 个 head（共 4 个，其中 2 个是**陪跑**——给第 4 节的 head patching 制造真实的定位问题）：

- **L1H0 · prev-token head**：QK 只看位置，$q_i \cdot k_j = \beta\,\mathbf{1}[j = i-1]$；OV 把被注意位置的 token 抄进 **KEY 块**。效果：每个 value 位置的 KEY 块 = 它配对的 key。
- **L1H1 · self head（陪跑）**：注意自己，把自己的 token 抄进 OUT——只影响 key token 的 logit，对 value 间的 logit diff 无影响。
- **L2H0 · recall head**：query 读当前 token（TOK 块），key 读 **KEY 块**（**K-composition**，模块 03）：$q_i \cdot k_j = \beta\,\mathbf{1}[t_i = t_{j-1}]$。末位置（token = `kq`）恰好注意到"前一个 token 是 `kq`"的位置——即 `kq` 的 value 位置；OV 把那里的 TOK（= value）抄进 OUT。
- **L2H1 · BOS head（陪跑）**：永远注意位置 0，给 BOS logit 加一个常数贡献。

`forward(tokens, hooks)` 暴露 5 个挂钩点（`resid_0/1/2`、`z_0/1`）——一个 30 行的微型 TransformerLens。**patching 的全部实现，就是在挂钩点把激活换掉**。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# ---- 词表与位置：key (1..7) 与 value (8..15) 取值区间不相交，匹配无歧义 ----
V    = 16                       # 0 = BOS, 1..7 = key, 8..15 = value
P    = 12                       # 最大序列长度
BETA = 16.0                     # QK 打分放大系数：softmax ≈ 硬注意力
sb   = np.sqrt(BETA)

# residual stream 块布局: [ TOK | POS | KEY | OUT ]
d   = V + P + V + V
TOK = slice(0, V)
POS = slice(V, V + P)
KEY = slice(V + P, V + P + V)
OUT = slice(V + P + V, d)

def head(Wq, Wk, Wv, Wo): return {"Wq": Wq, "Wk": Wk, "Wv": Wv, "Wo": Wo}

# ---- L1H0: prev-token head（i 注意 i-1，把 token 抄进 KEY 块）----
Wq = np.zeros((P, d)); Wq[:, POS] = sb * np.eye(P)
Wk = np.zeros((P, d)); Wk[1:, V:V + P - 1] = sb * np.eye(P - 1)   # 位置 p 的 key 放进槽位 p+1
Wv = np.zeros((V, d)); Wv[:, TOK] = np.eye(V)
Wo = np.zeros((d, V)); Wo[KEY, :] = np.eye(V)
L1H0 = head(Wq, Wk, Wv, Wo)

# ---- L1H1: self head（陪跑——注意自己，把自己的 token 抄进 OUT）----
Wq = np.zeros((P, d)); Wq[:, POS] = sb * np.eye(P)
Wk = np.zeros((P, d)); Wk[:, POS] = sb * np.eye(P)
Wv = np.zeros((V, d)); Wv[:, TOK] = np.eye(V)
Wo = np.zeros((d, V)); Wo[OUT, :] = 0.3 * np.eye(V)   # 缩放 0.3：陪跑贡献不盖过主电路
L1H1 = head(Wq, Wk, Wv, Wo)

# ---- L2H0: recall head（query 读 TOK，key 读 KEY 块 <- K-composition）----
Wq = np.zeros((V, d)); Wq[:, TOK] = sb * np.eye(V)
Wk = np.zeros((V, d)); Wk[:, KEY] = sb * np.eye(V)
Wv = np.zeros((V, d)); Wv[:, TOK] = np.eye(V)                     # 搬被注意位置的 token = value
Wo = np.zeros((d, V)); Wo[OUT, :] = np.eye(V)
L2H0 = head(Wq, Wk, Wv, Wo)

# ---- L2H1: BOS head（陪跑——永远注意位置 0）----
Wq = np.zeros((1, d)); Wq[0, POS] = sb                            # q_i = √β（任何位置都一样）
Wk = np.zeros((1, d)); Wk[0, V] = sb                              # 只有位置 0 的 key 非零
Wv = np.zeros((V, d)); Wv[:, TOK] = np.eye(V)
Wo = np.zeros((d, V)); Wo[OUT, :] = 0.3 * np.eye(V)   # 缩放 0.3，同上
L2H1 = head(Wq, Wk, Wv, Wo)

LAYERS = [[L1H0, L1H1], [L2H0, L2H1]]
W_U = np.zeros((V, d)); W_U[:, OUT] = np.eye(V)                   # unembedding 只读 OUT 块

def attn(x, h):
    qq, kk, vv = x @ h["Wq"].T, x @ h["Wk"].T, x @ h["Wv"].T
    n = x.shape[0]
    s = np.where(np.tril(np.ones((n, n), bool)), qq @ kk.T, -1e9) # causal mask
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    A = e / e.sum(axis=-1, keepdims=True)
    return (A @ vv) @ h["Wo"].T, A

def forward(tokens, hooks=None):
    """hooks: {挂钩点: fn}。挂钩点 resid_0/1/2 收 (L,d)，z_0/1 收 (n_heads,L,d)。"""
    hooks = hooks or {}
    n = len(tokens)
    x = np.zeros((n, d))
    x[np.arange(n), tokens] = 1.0                 # TOK 块
    x[np.arange(n), V + np.arange(n)] = 1.0       # POS 块
    if "resid_0" in hooks: x = hooks["resid_0"](x)
    cache = {"resid": [x.copy()], "z": [], "pattern": []}
    for li, layer in enumerate(LAYERS):
        outs = [attn(x, h) for h in layer]
        z = np.stack([o for o, _ in outs])        # (n_heads, L, d)：各 head 写回的贡献
        if f"z_{li}" in hooks: z = hooks[f"z_{li}"](z)
        x = x + z.sum(axis=0)
        if f"resid_{li+1}" in hooks: x = hooks[f"resid_{li+1}"](x)
        cache["resid"].append(x.copy())
        cache["z"].append(z)
        cache["pattern"].append(np.stack([A for _, A in outs]))
    return cache["resid"][-1] @ W_U.T, cache

def make_prompt(prs, query):
    """[BOS, k1, v1, k2, v2, ..., query_key]——模型应在末位置预测 query 配对的 value"""
    toks = [0]
    for k_, v_ in prs: toks += [k_, v_]
    return np.array(toks + [query])

def tok_name(t): return "BOS" if t == 0 else (f"k{t}" if t < 8 else f"v{t}")

# ---- 验证：零训练的手工电路解出键值召回 ----
demo = make_prompt([(1, 9), (2, 12), (3, 14)], query=2)
lg_demo, _ = forward(demo)
print("序列:", [tok_name(t) for t in demo])
print("末位置 top-1 预测:", tok_name(int(lg_demo[-1].argmax())), " (期望 v12)")
assert lg_demo[-1].argmax() == 12

# 键值对、query 全部随机重抽 200 次——电路是算法，不是记忆
ok = 0
for _ in range(200):
    ks = rng.permutation(np.arange(1, 8))[:3]
    vs = rng.permutation(np.arange(8, 16))[:3]
    qi = int(rng.integers(3))
    lg, _ = forward(make_prompt(list(zip(ks, vs)), query=int(ks[qi])))
    ok += int(lg[-1].argmax() == vs[qi])
print(f"200 次随机键值召回准确率: {ok / 200:.2%}")
assert ok == 200


## 2 · Clean / corrupted 最小对与 logit diff 度量

patching 实验 = 一个**最小对（minimal pair）** + 三次前向（讲解 §2）：① clean run 缓存全部激活；② corrupted run 给出行为被改变的基线；③ patched run 在 corrupted 的前向中植入 clean 的某个激活（**denoising** 方向），看行为恢复多少。

同一个 clean prompt（`k1→v9, k2→v12, k3→v14`，query = k1，答案 v9），先造两种 corruption：

| corruption | 改动 | 新答案 | 差异所在位置 |
|---|---|---|---|
| **换 value** | context 中 v9 → v11（pos 2） | v11 | 一个 context 位置 |
| **换 query** | 末位置 k1 → k3 | v14 | 只有末位置 |

度量用 **logit difference**（讲解 §3；Zhang & Nanda 2023 的推荐默认）：

$$\mathrm{LD} = \mathrm{logit}(t_{\text{ans}}) - \mathrm{logit}(t_{\text{alt}})$$

它是末层 residual stream 的**线性读出**（不会被 softmax 压扁）、远离概率度量的饱和区、且用 $t_{\text{alt}}$ 做双边对照（排除"只是整体抬高置信度"的混淆）。patched run 的效应归一化为

$$\mathrm{recovery} = \frac{\mathrm{LD}_{\text{patched}} - \mathrm{LD}_{\text{corr}}}{\mathrm{LD}_{\text{clean}} - \mathrm{LD}_{\text{corr}}}$$

**1 = 单独恢复这个激活就足以恢复行为，0 = 毫无作用。**


In [ ]:
# ---- 最小对：同一个 clean prompt，两种 corruption ----
pairs_c  = [(1, 9), (2, 12), (3, 14)]
clean    = make_prompt(pairs_c, query=1)                       # [BOS k1 v9 k2 v12 k3 v14 k1] -> v9
corr_val = make_prompt([(1, 11), (2, 12), (3, 14)], query=1)   # 换 value: pos 2 的 v9 -> v11
corr_qry = make_prompt(pairs_c, query=3)                       # 换 query: 末位置 k1 -> k3
POS_V, POS_Q = 2, len(clean) - 1

def logit_diff(logits, ans, alt):
    """LD = logit(ans) - logit(alt)，读末位置：线性、不饱和、双边对照（讲解 §3）"""
    return float(logits[-1, ans] - logits[-1, alt])

def recovery(ld_patched, ld_clean, ld_corr):
    return (ld_patched - ld_corr) / (ld_clean - ld_corr)

def run_with_resid_patch(base_tokens, point, pos, src_cache):
    """denoising：在 base_tokens 的前向中，把 resid_{point}[pos] 换成 src_cache 的对应激活"""
    src = src_cache["resid"][point]
    def hook(x, pos=pos, src=src):              # 默认参数绑定，防闭包陷阱
        x = x.copy(); x[pos] = src[pos]; return x
    lg, _ = forward(base_tokens, hooks={f"resid_{point}": hook})
    return lg

lg_clean, cache_clean = forward(clean)
print("clean:", [tok_name(t) for t in clean])
for name, corr, ans, alt in [("换 value", corr_val, 9, 11), ("换 query", corr_qry, 9, 14)]:
    lg_corr, _ = forward(corr)
    print(f"  {name}: LD_clean = {logit_diff(lg_clean, ans, alt):+.3f}   "
          f"LD_corr = {logit_diff(lg_corr, ans, alt):+.3f}   (ans={tok_name(ans)}, alt={tok_name(alt)})")

# ---- 单点 patch 尝鲜：换 value 的 corrupted run 里，把 resid_1 的某一行换回 clean ----
lg_cv, _ = forward(corr_val)
ld_c, ld_x = logit_diff(lg_clean, 9, 11), logit_diff(lg_cv, 9, 11)
print()
rs_demo = {}
for pos in [POS_V, 4]:
    lg_p = run_with_resid_patch(corr_val, 1, pos, cache_clean)
    rs_demo[pos] = recovery(logit_diff(lg_p, 9, 11), ld_c, ld_x)
    flag = "<- 关键位置：单独恢复它就足够" if pos == POS_V else "<- 无关位置：毫无作用"
    print(f"patch resid_1[pos {pos}] : recovery = {rs_demo[pos]:+.3f}   {flag}")
assert rs_demo[POS_V] > 0.99 and abs(rs_demo[4]) < 0.01


## 3 · 系统扫描：(layer × position) 因果热图

把单点 patch 扫过 **3 个挂钩点 × 全部 8 个位置**，每格一次 patched 前向，recovery 填成矩阵——这正是 causal tracing [Meng 2022] 的微缩版。先预测再跑：

- **换 value**：差异信息从 pos 2 的 embedding 出发，在 after L1 仍驻留在 pos 2（TOK 块无人改写），随后被 L2 recall head 搬到末位置——after L2 这一行应该**只有末位置亮**，pos 2 熄灭；
- **换 query**：两次输入只在末位置不同 → 整条因果路径竖在最后一列。

同一个任务、两种 corruption、两张完全不同的热图——patching 定位的从来不是"任务的电路"，而是**这对输入之间差异的流经之路**（讲解 §2.2）。


In [ ]:
def patching_heatmap(corr_tokens, src_cache, ld_clean, ld_corr, ans, alt):
    n = len(corr_tokens)
    R = np.zeros((3, n))
    for point in range(3):
        for pos in range(n):
            lg = run_with_resid_patch(corr_tokens, point, pos, src_cache)
            R[point, pos] = recovery(logit_diff(lg, ans, alt), ld_clean, ld_corr)
    return R

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.4))
for ax, (title, corr, ans, alt) in zip(axes, [("corrupt value (pos 2)", corr_val, 9, 11),
                                              ("corrupt query (pos 7)", corr_qry, 9, 14)]):
    lg_corr, _ = forward(corr)
    R = patching_heatmap(corr, cache_clean, logit_diff(lg_clean, ans, alt),
                         logit_diff(lg_corr, ans, alt), ans, alt)
    im = ax.imshow(R, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(corr)))
    ax.set_xticklabels([f"{i}\n{tok_name(t)}" for i, t in enumerate(corr)], fontsize=8)
    ax.set_yticks(range(3)); ax.set_yticklabels(["embed", "after L1", "after L2"])
    ax.set_title(f"denoising recovery -- {title}", fontsize=10)
    fig.colorbar(im, ax=ax, label="recovery")
plt.tight_layout(); plt.show()

print("左图：差异信息在 pos 2 驻留两层（embed、after L1 都亮），被 L2 recall head 搬走后")
print("      只在末位置可恢复——『先驻留、后搬运』的轨迹直接可见（causal tracing 微缩版）。")
print("右图：换 query 的差异只在末位置 → 整条因果路径竖在最后一列。")
print("口诀：corruption 没改的东西，patching 永远测不到（讲解 §2.2）。")


## 4 · Head-level patching：4 个 head 里谁是因果的

residual patching 回答"信息在哪层哪个位置"，head patching 回答"是**哪个组件**在搬"。挂钩点 `z_li` 暴露该层每个 head 写回 residual stream 的贡献（形状 `(n_heads, L, d)`）——把单个 head 的整条输出替换成 clean run 的值即可。

这次用第三种 corruption——**换 context key**（pos 1 的 k1 → k4，query 不变）：corrupted run 里没有任何 KEY 匹配 query，注意力均匀摊开、检索彻底失败（LD ≈ 0）。选它是因为差异信息必须**串行**流过

$$\text{L1H0（把 key 抄进 value 位置的 KEY 块）} \;\longrightarrow\; \text{L2H0（按 KEY 匹配检索 value）}$$

denoising 对串行路径上的**每个**节点都应给出 recovery ≈ 1（恢复任何一环都"足够"），而两个陪跑 head 应 ≈ 0。


In [ ]:
# 第三种 corruption：换 context key（pos 1 的 k1 -> k4），query 不变 -> 检索失败
corr_key = make_prompt([(4, 9), (2, 12), (3, 14)], query=1)
ANS, ALT = 9, 12                                  # alt 取 distractor 的 value v12
lg_ck, _ = forward(corr_key)
ld_c = logit_diff(lg_clean, ANS, ALT)
ld_x = logit_diff(lg_ck, ANS, ALT)
print(f"LD_clean = {ld_c:+.3f}（检索成功）   LD_corr = {ld_x:+.3f}（无匹配 -> 均匀注意力）\n")

def run_with_head_patch(base_tokens, layer, h_idx, src_cache):
    """denoising：把第 layer 层第 h_idx 个 head 的输出整条替换为 src_cache 的值"""
    src = src_cache["z"][layer]
    def hook(z, h_idx=h_idx, src=src):
        z = z.copy(); z[h_idx] = src[h_idx]; return z
    lg, _ = forward(base_tokens, hooks={f"z_{layer}": hook})
    return lg

names = [["L1H0 prev-token", "L1H1 self (陪跑)"], ["L2H0 recall   ", "L2H1 BOS (陪跑)"]]
R_head = np.zeros((2, 2))
for li in range(2):
    for hi in range(2):
        lg = run_with_head_patch(corr_key, li, hi, cache_clean)
        R_head[li, hi] = recovery(logit_diff(lg, ANS, ALT), ld_c, ld_x)
        print(f"patch {names[li][hi]} -> recovery = {R_head[li, hi]:+.3f}")

assert R_head[0, 0] > 0.9 and R_head[1, 0] > 0.9, "串行路径上的两个 head 都应足够"
assert abs(R_head[0, 1]) < 0.05 and abs(R_head[1, 1]) < 0.05, "陪跑 head 应无因果效应"
print("\n4 个 head -> 2 个因果 head，串成一条路径：")
print("  L1H0（把 key 抄进 value 位置的 KEY 块）-> L2H0（按 KEY 匹配把 value 搬到末位置）")
print("denoising 对串行路径上的每个节点都给 recovery ≈ 1：单独恢复任何一环都『足够』。")


## 5 · Noising ≠ Denoising：冗余电路上的不对称

把同一对 `k1→v9` 在 context 里**写两遍**（冗余），corruption 把两份拷贝的 key 都换成 k4。按讲解 §2.1 的表格预测：

- **denoising**（corrupted 里恢复单份拷贝的 key）：单份就足以让 recall head 找到匹配 → recovery ≈ **1.0**——充分性视角下两份拷贝"**都亮**"；
- **noising**（clean 里破坏单份拷贝的 key）：幸存拷贝顶上 → effect ≈ **0.0**——必要性视角下两份拷贝"**都不亮**"；只有同时破坏两份才致命。

机制就是 softmax 的**重归一化**：被破坏的 key 退出竞争后，注意力质量自动 100% 转移到幸存拷贝——这是 self-repair / hydra effect [McGrath 2023 arXiv:2307.15771] 的最小可复现实现。教训：**单点 ablation/noising 系统性低估冗余组件的重要性**（讲解 §7.1），下结论前 noising 与 denoising 必须分开报告。


In [ ]:
# 冗余电路：同一对 k1->v9 写两遍（key 在 pos 1 和 pos 3），外加 distractor k2->v12
clean_red = np.array([0, 1, 9, 1, 9, 2, 12, 1])
corr_red  = np.array([0, 4, 9, 4, 9, 2, 12, 1])    # 两份拷贝的 key 都换成 k4 -> 检索失败
COPY1, COPY2 = 1, 3
ANS, ALT = 9, 12

lg_cl, cache_cl = forward(clean_red)
lg_co, cache_co = forward(corr_red)
ld_cl, ld_co = logit_diff(lg_cl, ANS, ALT), logit_diff(lg_co, ANS, ALT)
print(f"LD_clean = {ld_cl:+.3f}（两份拷贝各分 ~50% 注意力，检索成功）")
print(f"LD_corr  = {ld_co:+.3f}（无匹配，检索失败）\n")

print("—— denoising：corrupted 里恢复单份拷贝的 key（问：它足够吗？）——")
den = []
for pos in [COPY1, COPY2]:
    lg = run_with_resid_patch(corr_red, 0, pos, cache_cl)
    den.append(recovery(logit_diff(lg, ANS, ALT), ld_cl, ld_co))
    print(f"  恢复拷贝 @ pos {pos}: recovery = {den[-1]:.3f}")

print("\n—— noising：clean 里破坏单份/两份拷贝的 key（问：它必要吗？）——")
noi = []
for pos in [COPY1, COPY2, [COPY1, COPY2]]:
    lg = run_with_resid_patch(clean_red, 0, pos, cache_co)       # 方向反转：src = corrupted
    noi.append((logit_diff(lg, ANS, ALT) - ld_cl) / (ld_co - ld_cl))  # 1 = 行为被完全破坏
    print(f"  破坏拷贝 @ pos {str(pos):6s}: effect = {noi[-1]:.3f}")

assert den[0] > 0.95 and den[1] > 0.95,           "denoising：单份拷贝各自都『足够』"
assert abs(noi[0]) < 0.05 and abs(noi[1]) < 0.05, "noising：单份拷贝都『不必要』"
assert noi[2] > 0.95,                             "同时破坏两份才致命"

# ---- 机制：softmax 重归一化 = self-repair ----
def l2h0_final_attn(tokens, hooks=None):
    _, c = forward(tokens, hooks=hooks)
    return c["pattern"][1][0][-1]                  # L2H0 在末位置的注意力分布

def noise_copy1(x, src=cache_co["resid"][0]):
    x = x.copy(); x[COPY1] = src[COPY1]; return x

A0 = l2h0_final_attn(clean_red)
A1 = l2h0_final_attn(clean_red, hooks={"resid_0": noise_copy1})
print("\nL2H0 末位置注意力（机制 = softmax 重归一化）：")
print("  clean          :", np.round(A0, 3))
print("  noising 拷贝 1 :", np.round(A1, 3), " <- 质量 100% 转移到幸存拷贝 (pos 4)")
assert abs(A0[2] - 0.5) < 0.01 and abs(A0[4] - 0.5) < 0.01
assert A1[4] > 0.99

print(f"\n同一位置 pos {COPY1}：denoising recovery = {den[0]:.2f}，noising effect = {noi[0]:.2f}")
print("-> 充分但不必要。这就是 self-repair 如何让 ablation 误导你（讲解 §7.1）。")


---
## ✏️ 练习 1：实现 `logit_diff_ex` 与 `recovery_ex`

不翻上文，自己实现 patching 的两件度量仪器：

- `logit_diff_ex(logits, ans, alt)`：读**末位置**，返回 `float` 的 `logit[ans] - logit[alt]`；
- `recovery_ex(ld_patched, ld_clean, ld_corr)`：把 patched LD 归一化——0 = 与 corrupted 相同，1 = 完全恢复 clean。

**提示**：各 1–2 行。`logits` 形状 `(L, V)`，末位置即 `logits[-1]`；交换 ans/alt 应使 LD 变号；recovery 的分母是 `ld_clean - ld_corr`（实验设计保证它远离 0，无需防御除零）。


In [ ]:
def logit_diff_ex(logits, ans, alt):
    # TODO: 末位置 logit(ans) - logit(alt)，返回 float
    raise NotImplementedError

def recovery_ex(ld_patched, ld_clean, ld_corr):
    # TODO: (ld_patched - ld_corr) / (ld_clean - ld_corr)
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
fake = np.zeros((3, V)); fake[-1, 9], fake[-1, 11] = 2.0, 0.5
assert abs(logit_diff_ex(fake, 9, 11) - 1.5) < 1e-9
assert abs(logit_diff_ex(fake, 11, 9) + 1.5) < 1e-9              # 交换 ans/alt -> 变号
assert logit_diff_ex(lg_clean, 9, 11) > 0.99                     # clean run：检索成功
assert abs(recovery_ex(1.0, 1.0, -1.0) - 1.0) < 1e-9             # 完全恢复
assert abs(recovery_ex(-1.0, 1.0, -1.0)) < 1e-9                  # 毫无作用
assert abs(recovery_ex(0.0, 1.0, -1.0) - 0.5) < 1e-9             # 恢复一半
print("✅ 练习 1 通过")


## ✏️ 练习 2：实现 `patch_resid_ex` —— 单点 residual patching

`patch_resid_ex(base_tokens, point, pos, src_cache, ans, alt)`：在 `base_tokens` 的前向中，把挂钩点 `resid_{point}` 的第 `pos` 行替换为 `src_cache` 同点同位置的激活，返回 patched run 的 **logit diff**（float）。

**提示**：约 6 行——构造 `hook(x)`（先 `x.copy()` 再替换第 `pos` 行），挂到 `forward(..., hooks={f"resid_{point}": hook})`，对返回的 logits 调 `logit_diff`。若以后在循环里批量造 hook，记得用默认参数绑定 `pos`（Python 闭包按引用捕获外层变量，这是 patching 代码最常见的 bug）。自测检查：关键位置恢复 > 90%、无关位置 ≈ 0、信息被搬走后原位置不再因果。


In [ ]:
def patch_resid_ex(base_tokens, point, pos, src_cache, ans, alt):
    # TODO:
    #  1) src = src_cache["resid"][point]
    #  2) hook(x)：复制 x，把第 pos 行替换为 src[pos]
    #  3) lg, _ = forward(base_tokens, hooks={f"resid_{point}": hook})
    #  4) 返回 logit_diff(lg, ans, alt)
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
lg_cv2, _ = forward(corr_val)
ldc2, ldx2 = logit_diff(lg_clean, 9, 11), logit_diff(lg_cv2, 9, 11)
r_key  = recovery(patch_resid_ex(corr_val, 1, POS_V, cache_clean, 9, 11), ldc2, ldx2)
r_off  = recovery(patch_resid_ex(corr_val, 1, 4, cache_clean, 9, 11), ldc2, ldx2)
r_late = recovery(patch_resid_ex(corr_val, 2, len(corr_val) - 1, cache_clean, 9, 11), ldc2, ldx2)
r_gone = recovery(patch_resid_ex(corr_val, 2, POS_V, cache_clean, 9, 11), ldc2, ldx2)
assert r_key > 0.9,        "关键位置（value 位）应恢复 > 90%"
assert abs(r_off) < 0.05,  "无关位置应 ≈ 0"
assert r_late > 0.9,       "after L2 的末位置应完全恢复（logits 直接读它）"
assert abs(r_gone) < 0.05, "信息已被搬到末位置后，原位置不再因果"
print("✅ 练习 2 通过")


## ✏️ 练习 3：实现 `patch_head_ex` —— head-level patching

`patch_head_ex(base_tokens, layer, h_idx, src_cache, ans, alt)`：把第 `layer` 层第 `h_idx` 个 head 写回 residual 的贡献（挂钩点 `z_{layer}`，形状 `(n_heads, L, d)`）整条替换为 `src_cache["z"][layer][h_idx]`，返回 patched run 的 logit diff。

**提示**：与练习 2 同构，约 6 行；hook 收到的是 `(n_heads, L, d)` 的 `z`，只替换 `z[h_idx]`。自测用"换 context key"的 corruption——串行因果路径 L1H0 → L2H0 上每个节点 denoising 都应 ≈ 1，两个陪跑 head ≈ 0。


In [ ]:
def patch_head_ex(base_tokens, layer, h_idx, src_cache, ans, alt):
    # TODO: hook(z) 只把 z[h_idx] 替换为 src_cache["z"][layer][h_idx]，
    #       挂到 f"z_{layer}"，返回 patched run 的 logit_diff
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
lg_ck3, _ = forward(corr_key)
ldc3, ldx3 = logit_diff(lg_clean, 9, 12), logit_diff(lg_ck3, 9, 12)
rs = {(li, hi): recovery(patch_head_ex(corr_key, li, hi, cache_clean, 9, 12), ldc3, ldx3)
      for li in range(2) for hi in range(2)}
assert rs[(0, 0)] > 0.9, "L1H0 prev-token head 在因果路径上"
assert rs[(1, 0)] > 0.9, "L2H0 recall head 在因果路径上"
assert abs(rs[(0, 1)]) < 0.05 and abs(rs[(1, 1)]) < 0.05, "陪跑 head 应 ≈ 0"
print("✅ 练习 3 通过")


## ✏️ 练习 4：实现 `patch_scan_ex` —— 系统性 patching 扫描

`patch_scan_ex(clean_tokens, corr_tokens, ans, alt)`：完整的扫描流水线——跑 clean / corrupted 两个 run 取基线 LD 与 clean cache，对每个 `(point ∈ {0,1,2}, pos ∈ {0,…,L-1})` 做一次 denoising patch，返回 `(3, L)` 的 recovery 矩阵（第 3 节热图的可复用版本）。

**提示**：约 10 行；可直接复用上文的 `run_with_resid_patch` 与 `recovery`（或你练习 2 的 `patch_resid_ex`）。自测会 assert 两种 corruption 下的**峰值位置**：换 value 时 embed / after-L1 两行的峰值都在 value 位置、after-L2 行的峰值在末位置；换 query 时所有行都竖在末位置。


In [ ]:
def patch_scan_ex(clean_tokens, corr_tokens, ans, alt):
    # TODO:
    #  1) 跑 clean / corrupted 两个 run，取 clean cache 与 LD_clean / LD_corr
    #  2) 对 point ∈ {0,1,2} × pos ∈ {0..L-1}：denoising patch 一次，算 recovery
    #  3) 返回 (3, L) recovery 矩阵
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
R_v = patch_scan_ex(clean, corr_val, 9, 11)
assert R_v.shape == (3, len(clean))
assert R_v[0].argmax() == POS_V and R_v[1].argmax() == POS_V     # 信息驻留在 value 位置
assert R_v[2].argmax() == len(clean) - 1                         # L2 之后已搬到末位置
assert R_v[1, POS_V] > 0.9 and abs(R_v[2, POS_V]) < 0.05
assert np.all(np.abs(np.delete(R_v[1], POS_V)) < 0.05)           # 其余位置 ≈ 0
R_q = patch_scan_ex(clean, corr_qry, 9, 14)
assert np.all(np.abs(R_q[:, :-1]) < 0.05) and np.all(R_q[:, -1] > 0.9)   # 整列竖在末位置
print("✅ 练习 4 通过")


---
## 📖 参考答案


In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def logit_diff_ex(logits, ans, alt):
    return float(logits[-1, ans] - logits[-1, alt])

def recovery_ex(ld_patched, ld_clean, ld_corr):
    return (ld_patched - ld_corr) / (ld_clean - ld_corr)


In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def patch_resid_ex(base_tokens, point, pos, src_cache, ans, alt):
    src = src_cache["resid"][point]
    def hook(x, pos=pos, src=src):              # 默认参数绑定，防闭包陷阱
        x = x.copy(); x[pos] = src[pos]; return x
    lg, _ = forward(base_tokens, hooks={f"resid_{point}": hook})
    return logit_diff(lg, ans, alt)


In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def patch_head_ex(base_tokens, layer, h_idx, src_cache, ans, alt):
    src = src_cache["z"][layer]
    def hook(z, h_idx=h_idx, src=src):
        z = z.copy(); z[h_idx] = src[h_idx]; return z
    lg, _ = forward(base_tokens, hooks={f"z_{layer}": hook})
    return logit_diff(lg, ans, alt)


In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def patch_scan_ex(clean_tokens, corr_tokens, ans, alt):
    lg_c, cache_c = forward(clean_tokens)
    lg_x, _ = forward(corr_tokens)
    ld_c, ld_x = logit_diff(lg_c, ans, alt), logit_diff(lg_x, ans, alt)
    n = len(corr_tokens)
    R = np.zeros((3, n))
    for point in range(3):
        for pos in range(n):
            lg = run_with_resid_patch(corr_tokens, point, pos, cache_c)
            R[point, pos] = recovery(logit_diff(lg, ans, alt), ld_c, ld_x)
    return R


---
## 🎯 真实数据胶囊题：真实 embedding 上的 activation patching（因果归因）

patching 把一个表示的某些维度替换成另一个，看输出怎么变——定位“哪些维度负责某概念”。用真实 GPT-2 embedding + 数字 probe：把字母 token 的 top 维度替换成数字方向，看 probe 的“数字分”上升。

> 本模块新增的**真实数据**练习：用**真实 GPT-2 权重/embedding**把本章的可解释性技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, struct, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.interp_data"); os.makedirs(CACHE,exist_ok=True)
ST="https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
def _rng(s,e):
    req=urllib.request.Request(ST, headers={"Range":f"bytes={s}-{e}"})
    return urllib.request.urlopen(req,timeout=60).read()
def gpt2_emb_block(n=6000):
    cache=os.path.join(CACHE,f"wte_{n}.npy")
    if os.path.exists(cache): return np.load(cache)
    hlen=struct.unpack("<Q", _rng(0,7))[0]; hdr=json.loads(_rng(8,8+hlen-1))
    info=hdr["wte.weight"]; base=8+hlen; s0=info["data_offsets"][0]; d=info["shape"][1]
    raw=_rng(base+s0, base+s0+n*d*4-1)
    E=np.frombuffer(raw,dtype=np.float32).reshape(n,d).copy()
    np.save(cache,E); return E
def gpt2_vocab():
    p=os.path.join(CACHE,"vocab.json")
    if not os.path.exists(p): urllib.request.urlretrieve("https://huggingface.co/openai-community/gpt2/resolve/main/vocab.json",p)
    return json.load(open(p))
def digit_letter_dataset(lim=6000):
    "返回 (X[token嵌入], y[1=数字 0=字母], E, ids_digit, ids_alpha)"
    v=gpt2_vocab(); E=gpt2_emb_block(lim)
    dig=[i for t,i in v.items() if i<lim and t.isdigit()]
    alpha=[i for t,i in v.items() if i<lim and t.isalpha() and t.isascii()]
    rng=np.random.default_rng(0); alpha=list(rng.permutation(alpha)[:len(dig)])
    ids=dig+alpha; y=np.array([1]*len(dig)+[0]*len(alpha))
    return E[ids], y, E, dig, alpha
def shakespeare():
    p=os.path.join(CACHE,"shake.txt")
    if not os.path.exists(p): urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",p)
    return open(p).read()

X,y,E,dig,alpha=digit_letter_dataset()
# 先训一个数字 probe(标准化)
mu,sd=X.mean(0),X.std(0)+1e-8; Xs=(X-mu)/sd
w=np.zeros(X.shape[1]); b=0.0
for _ in range(600):
    p=1/(1+np.exp(-(Xs@w+b))); g=p-y; w-=0.5*Xs.T@g/len(y); b-=0.5*g.mean()
def digit_score(vec): return float(1/(1+np.exp(-(((vec-mu)/sd)@w+b))))
print("数字方向 probe 训好；字母 token 平均数字分:", np.mean([digit_score(E[a]) for a in alpha]).round(3))

**练习**：实现 `patch(target, source, dims)`：把 `target` 向量在 `dims` 这些维度上替换成 `source` 的值，返回新向量。用它把字母 token 在“probe 权重最大的若干维”上 patch 成数字 token 的均值，验证数字分上升。

In [ ]:
def patch(target, source, dims):
    # TODO: 拷贝 target，在 dims 维度上用 source 的值覆盖，返回
    raise NotImplementedError


In [ ]:
# 自测
dig_mean=E[dig].mean(0)
top_dims=np.argsort(-np.abs(w))[:50]   # probe 最依赖的维度
before=np.mean([digit_score(E[a]) for a in alpha[:30]])
after =np.mean([digit_score(patch(E[a], dig_mean, top_dims)) for a in alpha[:30]])
assert after > before, "patch 数字方向维度后，字母 token 的数字分应上升"
print(f"activation patching ✓  字母token数字分 {before:.3f} -> patch后 {after:.3f}")


### 📖 参考答案

In [ ]:
def patch(target, source, dims):
    out=target.copy(); out[dims]=source[dims]; return out
print("✓ patching 是因果归因：换掉某些维度看输出变化，定位概念所在")